# Phase 2 — Preprocessing & Feature Engineering

### Objectifs
1. Pipeline sklearn (imputation, encoding, scaling)
2. Feature engineering (ratios, interactions)
3. Split train/test stratifié

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import joblib
import os

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder, OrdinalEncoder
from sklearn.impute import SimpleImputer

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid', palette='muted')

# Créer les dossiers nécessaires
os.makedirs('../reports/figures', exist_ok=True)
os.makedirs('../models', exist_ok=True)
os.makedirs('../data', exist_ok=True)

FIGURES_DIR = '../reports/figures/'
MODELS_DIR  = '../models/'
DATA_DIR    = '../data/'

df = pd.read_csv('../customer_churn.csv')
print(f'Dataset chargé : {df.shape}')
df.head()

## 1. Feature Engineering

Création de nouvelles variables dérivées à partir des constats de l'EDA :
- **Redondance** : `total_revenue ≈ monthly_fee × tenure_months` → créer `revenue_per_month`
- **Ratios comportementaux** : rapporter les tickets/escalades à l'ancienneté du client
- **Signaux de désengagement** : combiner les variables d'activité

Comme le sujet l'indique : *une variable dérivée peut être plus pertinente qu'une variable brute.*

In [ ]:
df_fe = df.copy()

# --- Ratios financiers ---
# Evite division par zéro avec np.where
df_fe['revenue_per_month'] = np.where(
    df_fe['tenure_months'] > 0,
    df_fe['total_revenue'] / df_fe['tenure_months'],
    df_fe['monthly_fee']
)

# --- Ratios support ---
df_fe['tickets_per_month'] = np.where(
    df_fe['tenure_months'] > 0,
    df_fe['support_tickets'] / df_fe['tenure_months'],
    0
)

df_fe['escalation_rate'] = np.where(
    df_fe['support_tickets'] > 0,
    df_fe['escalations'] / df_fe['support_tickets'],
    0
)

# --- Score d'engagement (combinaison de signaux d'activité) ---
# Normalisation min-max manuelle pour chaque composante
def minmax(s):
    return (s - s.min()) / (s.max() - s.min() + 1e-9)

df_fe['engagement_score'] = (
    minmax(df_fe['monthly_logins']) +
    minmax(df_fe['weekly_active_days']) +
    minmax(df_fe['features_used']) -
    minmax(df_fe['last_login_days_ago'])
) / 4

# --- Signal de risque financier ---
df_fe['financial_risk'] = (
    minmax(df_fe['payment_failures']) +
    (1 - minmax(df_fe['csat_score']))
) / 2

new_features = ['revenue_per_month', 'tickets_per_month', 'escalation_rate',
                'engagement_score', 'financial_risk']

print('Nouvelles features créées :')
print(df_fe[new_features].describe().round(3))

## 2. Corrélation des nouvelles features avec le Churn

In [ ]:
all_num = ['age', 'tenure_months', 'monthly_logins', 'weekly_active_days',
           'avg_session_time', 'features_used', 'usage_growth_rate',
           'last_login_days_ago', 'monthly_fee', 'total_revenue',
           'payment_failures', 'support_tickets', 'avg_resolution_time',
           'csat_score', 'escalations', 'email_open_rate',
           'marketing_click_rate', 'nps_score', 'referral_count'] + new_features

corr_churn = df_fe[all_num].corrwith(df_fe['churn']).sort_values()

fig, ax = plt.subplots(figsize=(10, 10))
colors = ['#e74c3c' if v > 0 else '#2ecc71' for v in corr_churn.values]
bars = ax.barh(corr_churn.index, corr_churn.values, color=colors)

# Mettre en évidence les nouvelles features
for i, (name, val) in enumerate(zip(corr_churn.index, corr_churn.values)):
    if name in new_features:
        bars[i].set_edgecolor('black')
        bars[i].set_linewidth(2)
    ax.text(val + (0.002 if val > 0 else -0.002), i, f'{val:.3f}',
            va='center', ha='left' if val > 0 else 'right', fontsize=8)

ax.set_xlabel('Corrélation avec Churn')
ax.set_title('Corrélation avec le Churn — features originales + engineered (bordure noire)', fontsize=12, fontweight='bold')
ax.axvline(x=0, color='black', linestyle='-', linewidth=0.5)
plt.tight_layout()
plt.savefig(f'{FIGURES_DIR}correlation_churn_fe.png', bbox_inches='tight')
plt.show()

print('Features engineered vs features originales :')
orig_max = df_fe[['csat_score','payment_failures','tenure_months']].corrwith(df_fe['churn']).abs().max()
fe_max = df_fe[new_features].corrwith(df_fe['churn']).abs().max()
print(f'  Max |corrélation| originale : {orig_max:.3f}')
print(f'  Max |corrélation| engineered: {fe_max:.3f}')

## 3. Suppression des variables non pertinentes

D'après l'EDA :
- `customer_id` : identifiant unique, aucune valeur prédictive → **supprimer**
- `total_revenue` : redondant avec `monthly_fee × tenure_months` → **supprimer** (remplacé par `revenue_per_month`)

In [ ]:
cols_to_drop = ['customer_id', 'total_revenue']
df_clean = df_fe.drop(columns=cols_to_drop)

print(f'Shape avant suppression : {df_fe.shape}')
print(f'Shape après suppression  : {df_clean.shape}')
print(f'Colonnes supprimées : {cols_to_drop}')

## 4. Définition des colonnes par type

In [ ]:
TARGET = 'churn'

# Colonnes catégorielles
cat_cols_onehot = ['gender', 'signup_channel', 'payment_method', 'complaint_type', 'survey_response']
cat_cols_ordinal = ['customer_segment', 'contract_type', 'discount_applied', 'price_increase_last_3m']

# Toutes les colonnes numériques (originales + engineered)
num_cols = [c for c in df_clean.columns
            if c not in cat_cols_onehot + cat_cols_ordinal + [TARGET]]

print(f'Numériques ({len(num_cols)}) : {num_cols}')
print(f'\nCatégorielles OneHot ({len(cat_cols_onehot)}) : {cat_cols_onehot}')
print(f'\nCatégorielles Ordinal ({len(cat_cols_ordinal)}) : {cat_cols_ordinal}')

## 5. Split Train / Test stratifié

Le split **stratifié** garantit que le ratio churn (~10%) est identique dans train et test.
Sans stratification, le test set pourrait par hasard contenir trop peu de churners.

> **Important** : le preprocessing (imputation, scaling) sera fitté **uniquement sur le train set**
> pour éviter le **data leakage** — fuite d'information du test vers le train.

In [ ]:
X = df_clean.drop(columns=[TARGET])
y = df_clean[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print(f'Train : {X_train.shape} | Churn rate: {y_train.mean():.1%}')
print(f'Test  : {X_test.shape}  | Churn rate: {y_test.mean():.1%}')
print('\n=> Les taux de churn sont bien équilibrés entre train et test (stratification ok)')

## 6. Construction du Pipeline sklearn

Le pipeline enchaîne automatiquement :
1. **Imputation** : remplace les valeurs manquantes
   - Numérique → médiane (robuste aux outliers)
   - Catégorielle → valeur la plus fréquente (ou `No_Complaint` pour complaint_type)
2. **Encoding** : convertit les catégories en nombres
   - OneHotEncoder pour les variables nominales (sans ordre)
   - OrdinalEncoder pour les variables ordinales (avec ordre)
3. **Scaling** : StandardScaler pour les numériques (moyenne=0, écart-type=1)
   - Nécessaire pour la Régression Logistique
   - Sans effet sur Random Forest / XGBoost (mais ne nuit pas)

Avantage du Pipeline : évite le **data leakage** car `fit` se fait uniquement sur le train.

In [ ]:
# Sous-pipeline numérique
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler',  StandardScaler())
])

# Sous-pipeline catégoriel OneHot
onehot_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='constant', fill_value='No_Complaint')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', sparse_output=False))
])

# Sous-pipeline catégoriel Ordinal
ordinal_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OrdinalEncoder(handle_unknown='use_encoded_value', unknown_value=-1))
])

# Assemblage ColumnTransformer
preprocessor = ColumnTransformer([
    ('num',     numeric_pipeline,  num_cols),
    ('onehot',  onehot_pipeline,   cat_cols_onehot),
    ('ordinal', ordinal_pipeline,  cat_cols_ordinal)
], remainder='drop')

print('Pipeline créé :')
print(preprocessor)

## 7. Fit du préprocesseur sur le train set

In [ ]:
# Fit UNIQUEMENT sur le train — le test reste invisible
preprocessor.fit(X_train)

X_train_processed = preprocessor.transform(X_train)
X_test_processed  = preprocessor.transform(X_test)

print(f'Shape X_train transformé : {X_train_processed.shape}')
print(f'Shape X_test  transformé : {X_test_processed.shape}')

# Récupérer les noms des features après transformation
onehot_feature_names = preprocessor.named_transformers_['onehot']['encoder'].get_feature_names_out(cat_cols_onehot)
all_feature_names = num_cols + list(onehot_feature_names) + cat_cols_ordinal

print(f'\nNombre total de features après encoding : {len(all_feature_names)}')

## 8. Vérification visuelle du scaling

In [ ]:
# Vérifier que les numériques sont bien centrées-réduites
X_train_df = pd.DataFrame(X_train_processed, columns=all_feature_names)

num_check = X_train_df[num_cols[:6]].describe().round(3)
print('Statistiques des 6 premières features numériques après scaling :')
print(num_check)
print('\n=> mean ≈ 0, std ≈ 1 : scaling correct')

## 9. Sauvegarde

In [ ]:
# Sauvegarder le preprocesseur
joblib.dump(preprocessor, f'{MODELS_DIR}preprocessor.joblib')
print(f'Preprocesseur sauvegardé : {MODELS_DIR}preprocessor.joblib')

# Sauvegarder les splits (format numpy pour compatibilité)
np.save(f'{DATA_DIR}X_train.npy', X_train_processed)
np.save(f'{DATA_DIR}X_test.npy',  X_test_processed)
np.save(f'{DATA_DIR}y_train.npy', y_train.values)
np.save(f'{DATA_DIR}y_test.npy',  y_test.values)

# Sauvegarder les noms de features
import json
with open(f'{DATA_DIR}feature_names.json', 'w') as f:
    json.dump(all_feature_names, f)

print('Splits sauvegardés :')
print(f'  {DATA_DIR}X_train.npy  — {X_train_processed.shape}')
print(f'  {DATA_DIR}X_test.npy   — {X_test_processed.shape}')
print(f'  {DATA_DIR}y_train.npy  — {y_train.shape}')
print(f'  {DATA_DIR}y_test.npy   — {y_test.shape}')
print(f'  {DATA_DIR}feature_names.json — {len(all_feature_names)} features')

## 10. Synthèse Phase 2

| Étape | Action | Justification |
|-------|--------|---------------|
| **Feature Engineering** | 5 nouvelles features (ratios, scores) | Capturer des relations non-linéaires, réduire redondance |
| **Suppression** | `customer_id`, `total_revenue` | Pas de valeur prédictive / redondant |
| **Split stratifié** | 80% train / 20% test | Conserver le ratio churn dans chaque split |
| **Imputation** | Médiane (num), constante (cat) | Robuste aux outliers, encode l'absence de plainte |
| **Encoding** | OneHot (nominal), Ordinal (ordinal) | Adapté au type de variable |
| **Scaling** | StandardScaler | Nécessaire pour Logistic Regression |
| **Pipeline** | sklearn Pipeline + ColumnTransformer | Évite le data leakage, reproductible |

### Prochaine étape — Phase 3 : Modélisation
Les données sont prêtes. On peut maintenant entraîner :
1. **Logistic Regression** (baseline)
2. **Random Forest**
3. **XGBoost / LightGBM**